In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, roc_auc_score, roc_curve
)
import warnings
warnings.filterwarnings("ignore")

In [2]:
df = pd.read_csv("diabetes.csv")

print("=" * 60)
print("DIABETES PREDICTION - CAPSTONE PROJECT")
print("=" * 60)

DIABETES PREDICTION - CAPSTONE PROJECT


In [3]:
print("\n[1] Dataset Shape:", df.shape)
print("\n[2] First 5 Rows:")
print(df.head())
print("\n[3] Dataset Info:")
print(df.info())
print("\n[4] Basic Statistics:")
print(df.describe())
print("\n[5] Missing Values:")
print(df.isnull().sum())
print("\n[6] Target Distribution:")
print(df["Outcome"].value_counts())



[1] Dataset Shape: (768, 9)

[2] First 5 Rows:
   Pregnancies  Glucose  BloodPressure  SkinThickness  Insulin   BMI  \
0            6      148             72             35        0  33.6   
1            1       85             66             29        0  26.6   
2            8      183             64              0        0  23.3   
3            1       89             66             23       94  28.1   
4            0      137             40             35      168  43.1   

   DiabetesPedigreeFunction  Age  Outcome  
0                     0.627   50        1  
1                     0.351   31        0  
2                     0.672   32        1  
3                     0.167   21        0  
4                     2.288   33        1  

[3] Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 768 entries, 0 to 767
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Pregnancies      

In [4]:
zero_cols = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]

# Replace 0s with NaN and then fill with column median
for col in zero_cols:
    df[col] = df[col].replace(0, np.nan)
    df[col] = df[col].fillna(df[col].median())

print("\n[7] After cleaning - Missing Values:")
print(df.isnull().sum())


[7] After cleaning - Missing Values:
Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64


In [5]:
# --- 5. Exploratory Data Analysis (EDA) ---

# 5a. Correlation Heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(df.corr(), annot=True, fmt=".2f", cmap="Blues")
plt.title("Feature Correlation Heatmap")
plt.tight_layout()
plt.savefig("correlation_heatmap.png", dpi=150)
plt.close()
print("\n[8] Saved: correlation_heatmap.png")

# 5b. Outcome Distribution
plt.figure(figsize=(5, 4))
df["Outcome"].value_counts().plot(kind="bar", color=["steelblue", "tomato"])
plt.title("Outcome Distribution (0=No Diabetes, 1=Diabetes)")
plt.xlabel("Outcome")
plt.ylabel("Count")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig("outcome_distribution.png", dpi=150)
plt.close()
print("[9] Saved: outcome_distribution.png")

# 5c. Feature Distributions by Outcome
features = ["Glucose", "BMI", "Age", "Insulin"]
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for i, feat in enumerate(features):
    df.groupby("Outcome")[feat].plot(kind="hist", alpha=0.6, bins=20, ax=axes[i], legend=True)
    axes[i].set_title(feat)
    axes[i].set_xlabel(feat)
fig.suptitle("Feature Distributions by Outcome (Blue=0, Orange=1)", y=1.02)
plt.tight_layout()
plt.savefig("feature_distributions.png", dpi=150)
plt.close()
print("[10] Saved: feature_distributions.png")


[8] Saved: correlation_heatmap.png
[9] Saved: outcome_distribution.png
[10] Saved: feature_distributions.png


In [6]:
# --- 6. Feature & Target Split ---
X = df.drop("Outcome", axis=1)
y = df["Outcome"]

In [7]:
# --- 7. Train-Test Split ---
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"\n[11] Train size: {X_train.shape[0]}, Test size: {X_test.shape[0]}")


[11] Train size: 614, Test size: 154


In [8]:
# --- 8. Feature Scaling ---
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

In [9]:
# --- 9. Model Training ---
models = {
    "Logistic Regression": LogisticRegression(random_state=42),
    "Decision Tree":       DecisionTreeClassifier(random_state=42, max_depth=5),
    "Random Forest":       RandomForestClassifier(random_state=42, n_estimators=100)
}

results = {}

print("\n" + "=" * 60)
print("MODEL EVALUATION")
print("=" * 60)

for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    y_prob = model.predict_proba(X_test_scaled)[:, 1]

    acc  = accuracy_score(y_test, y_pred)
    auc  = roc_auc_score(y_test, y_prob)
    cv   = cross_val_score(model, X_train_scaled, y_train, cv=5, scoring="accuracy").mean()

    results[name] = {"Accuracy": acc, "AUC": auc, "CV Accuracy": cv,
                     "y_pred": y_pred, "y_prob": y_prob}

    print(f"\n--- {name} ---")
    print(f"  Test Accuracy : {acc:.4f}")
    print(f"  ROC-AUC Score : {auc:.4f}")
    print(f"  CV Accuracy   : {cv:.4f}")
    print("\n  Classification Report:")
    print(classification_report(y_test, y_pred, target_names=["No Diabetes", "Diabetes"]))


MODEL EVALUATION

--- Logistic Regression ---
  Test Accuracy : 0.7078
  ROC-AUC Score : 0.8130
  CV Accuracy   : 0.7818

  Classification Report:
              precision    recall  f1-score   support

 No Diabetes       0.75      0.82      0.78       100
    Diabetes       0.60      0.50      0.55        54

    accuracy                           0.71       154
   macro avg       0.68      0.66      0.67       154
weighted avg       0.70      0.71      0.70       154


--- Decision Tree ---
  Test Accuracy : 0.7597
  ROC-AUC Score : 0.7610
  CV Accuracy   : 0.7231

  Classification Report:
              precision    recall  f1-score   support

 No Diabetes       0.84      0.78      0.81       100
    Diabetes       0.64      0.72      0.68        54

    accuracy                           0.76       154
   macro avg       0.74      0.75      0.74       154
weighted avg       0.77      0.76      0.76       154


--- Random Forest ---
  Test Accuracy : 0.7792
  ROC-AUC Score : 0.8179
 

In [10]:
# --- 10. Confusion Matrices ---
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (name, res) in zip(axes, results.items()):
    cm = confusion_matrix(y_test, res["y_pred"])
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
                xticklabels=["No Diabetes", "Diabetes"],
                yticklabels=["No Diabetes", "Diabetes"])
    ax.set_title(f"{name}\nAccuracy: {res['Accuracy']:.2%}")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
plt.tight_layout()
plt.savefig("confusion_matrices.png", dpi=150)
plt.close()
print("\n[12] Saved: confusion_matrices.png")


[12] Saved: confusion_matrices.png


In [11]:
# --- 11. ROC Curves ---
plt.figure(figsize=(7, 5))
for name, res in results.items():
    fpr, tpr, _ = roc_curve(y_test, res["y_prob"])
    plt.plot(fpr, tpr, label=f"{name} (AUC={res['AUC']:.2f})")
plt.plot([0, 1], [0, 1], "k--", label="Random Guess")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves - All Models")
plt.legend()
plt.tight_layout()
plt.savefig("roc_curves.png", dpi=150)
plt.close()
print("[13] Saved: roc_curves.png")

[13] Saved: roc_curves.png


In [12]:
# --- 12. Feature Importance (Random Forest) ---
rf_model = models["Random Forest"]
importances = pd.Series(rf_model.feature_importances_, index=X.columns).sort_values(ascending=True)

plt.figure(figsize=(7, 5))
importances.plot(kind="barh", color="steelblue")
plt.title("Feature Importance - Random Forest")
plt.xlabel("Importance Score")
plt.tight_layout()
plt.savefig("feature_importance.png", dpi=150)
plt.close()
print("[14] Saved: feature_importance.png")

[14] Saved: feature_importance.png


In [14]:
# --- 13. Model Comparison Summary ---
print("\n" + "=" * 60)
print("MODEL COMPARISON SUMMARY")
print("=" * 60)
summary = pd.DataFrame({
    name: {"Accuracy": f"{res['Accuracy']:.4f}",
           "AUC":      f"{res['AUC']:.4f}",
           "CV Acc":   f"{res['CV Accuracy']:.4f}"}
    for name, res in results.items()
}).T
print(summary.to_string())

best_model = max(results, key=lambda k: results[k]["AUC"])
print(f"\n✓ Best Model by AUC: {best_model} (AUC = {results[best_model]['AUC']:.4f})")

print("\n[DONE] All outputs saved. Check PNG files for visualizations.")


MODEL COMPARISON SUMMARY
                    Accuracy     AUC  CV Acc
Logistic Regression   0.7078  0.8130  0.7818
Decision Tree         0.7597  0.7610  0.7231
Random Forest         0.7792  0.8179  0.7688

✓ Best Model by AUC: Random Forest (AUC = 0.8179)

[DONE] All outputs saved. Check PNG files for visualizations.
